# 00 · Inicio en Google Colab — Proyecto Estadísticas de YouTube

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<usuario>/youtube-stats-nlp/blob/main/notebooks/00_colab_inicio.ipynb)

Este notebook prepara el entorno y ejecuta la etapa de datos de principio a fin:

1. Clona el repositorio de GitHub e instala lo que Colab no trae.
2. Monta Google Drive para guardar datos y reportes (Colab borra el disco al cerrar la sesión).
3. Carga las credenciales desde los **Secretos** de Colab (nunca escritas en el notebook).
4. Ejecuta el pipeline con la muestra, con Kaggle y con la YouTube Data API.
5. Muestra los reportes y, opcionalmente, sube cambios de código a GitHub.

**Antes de empezar**, en el panel izquierdo de Colab abra 🔑 *Secretos* y cree los que necesite, activando *Acceso del notebook*:

| Secreto | Para qué | Dónde se obtiene |
| --- | --- | --- |
| `KAGGLE_USERNAME`, `KAGGLE_KEY` | Descargar el dataset de Kaggle | kaggle.com → Settings → API → Create New Token |
| `YOUTUBE_API_KEY` | Extraer datos con la YouTube Data API v3 | Google Cloud Console → APIs y servicios → Credenciales |
| `GITHUB_TOKEN` | Clonar un repo privado y hacer *push* | GitHub → Settings → Developer settings → Fine-grained tokens (permiso *Contents: Read and write*) |

No se necesita GPU para esta etapa: *Entorno de ejecución → Cambiar tipo* → CPU.

In [ ]:
# @title 1 · Clonar el repositorio e instalar dependencias
REPO_URL = "https://github.com/<usuario>/youtube-stats-nlp.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import base64, os, subprocess, sys

def _auth_args():
    """Encabezado de autenticación si existe el secreto GITHUB_TOKEN (repos privados)."""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        return []
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return ["-c", f"http.extraHeader=Authorization: Basic {auth}"]

def _git(*args):
    r = subprocess.run(["git", *_auth_args(), *args], capture_output=True, text=True)
    if r.returncode != 0:  # no se imprime el comando: puede contener el token
        raise RuntimeError("git " + args[0] + " falló:\n" + r.stderr[-500:])

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if "<usuario>" in REPO_URL:
        raise ValueError("Cambie REPO_URL por la URL de su repositorio en GitHub")
    REPO_DIR = "/content/youtube-stats-nlp"
    if os.path.exists(REPO_DIR):
        _git("-C", REPO_DIR, "pull", "-q", "origin", BRANCH)
    else:
        _git("clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_DIR}/requirements-colab.txt"], check=True)
    os.chdir(REPO_DIR)
else:  # computador local: el notebook se abre desde la carpeta notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO_DIR)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Repositorio listo en", REPO_DIR)

In [ ]:
# @title 2 · Montar Drive y cargar secretos
USAR_DRIVE = True  # @param {type:"boolean"}
CARPETA_DRIVE = "MyDrive/ytnlp-proyecto"  # @param {type:"string"}

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from ytnlp.colab import setup
info = setup(use_drive=USAR_DRIVE, drive_folder=CARPETA_DRIVE)

## 3 · Verificar que todo funciona

Corre las pruebas automáticas y el pipeline completo sobre una muestra **sintética** (no requiere credenciales; sus resultados no sirven para conclusiones).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], check=False)
subprocess.run([sys.executable, "scripts/make_sample_data.py"], check=True)

from ytnlp.pipeline import run
run("sample")

## 4 · Datos reales de Kaggle

Requiere los secretos `KAGGLE_USERNAME` y `KAGGLE_KEY`. El dataset se descarga una sola vez a Drive.

In [ ]:
import os
from ytnlp.pipeline import run
if not (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY")):
    print("Faltan los secretos de Kaggle; ver la tabla al inicio del notebook.")
else:
    print(run("kaggle"))

## 5 · Ver los reportes

Los archivos quedan en la carpeta de Drive configurada (`reports/`), así que también se pueden abrir desde Google Drive.

In [ ]:
# @title Mostrar reportes
REPORTE = "quality_report.md"  # @param ["quality_report.md", "stats_report.md", "baseline_report.md"]
from IPython.display import Image, Markdown, display
from ytnlp.config import path

rep = path("reports")
texto = (rep / REPORTE).read_text(encoding="utf-8")
display(Markdown(texto.split("## Figuras")[0]))
if REPORTE == "stats_report.md":
    for img in sorted((rep / "figures").glob("*.png")):
        display(Image(filename=str(img), width=520))

## 6 · Ampliar los datos con la YouTube Data API

Toma los Video IDs del dataset de Kaggle y descarga, por video, estadísticas actualizadas, duración, suscriptores del canal y hasta 100 comentarios. Cada video cuesta pocas unidades de cuota (`videos.list` y `commentThreads.list` = 1 unidad por llamada), así que conviene empezar con un límite pequeño. La extracción queda en Drive en `data/raw/api/<fecha>/`.

In [ ]:
# @title Extraer con la API
LIMITE_VIDEOS = 50  # @param {type:"integer"}
import os
if not os.getenv("YOUTUBE_API_KEY"):
    print("Falta el secreto YOUTUBE_API_KEY.")
else:
    from ytnlp.data.kaggle_source import load
    from ytnlp.data.youtube_api import extract
    ids = load()[0]["video_id"].dropna().unique().tolist()[:LIMITE_VIDEOS]
    carpeta = extract(ids)
    print("Extracción guardada en", carpeta)
    from ytnlp.pipeline import run
    run("api")

## 7 · Guardar cambios de código en GitHub (opcional)

Si modificó archivos del repositorio desde Colab (por ejemplo `configs/config.yaml` o algo en `src/`), esta celda hace *commit* y *push* con el secreto `GITHUB_TOKEN`. Los datos y reportes **no** se suben: viven en Drive y están en `.gitignore`.

Para guardar un notebook en GitHub use *Archivo → Guardar una copia en GitHub*.

In [ ]:
# @title Commit y push
HACER_PUSH = False  # @param {type:"boolean"}
MENSAJE = "Actualiza configuración desde Colab"  # @param {type:"string"}
NOMBRE = ""  # @param {type:"string"}
CORREO = ""  # @param {type:"string"}
if HACER_PUSH:
    from ytnlp.colab import git_push
    git_push(MENSAJE, branch=BRANCH, name=NOMBRE or None, email=CORREO or None)
else:
    print("Marque HACER_PUSH para subir los cambios.")

## Notas sobre Colab

- **La sesión es temporal.** Se desconecta tras un tiempo de inactividad o al llegar al máximo de horas; el disco `/content` se borra. Por eso el código se versiona en GitHub y los datos se guardan en Drive.
- **Al volver**, basta con ejecutar las celdas 1 y 2: el repo se vuelve a clonar (o se actualiza con `git pull`) y los datos siguen en Drive.
- **Trabajo en equipo:** cada integrante abre el notebook desde GitHub con el botón *Abrir en Colab*, usa sus propios secretos y una carpeta compartida de Drive: quien la recibe la agrega como *acceso directo en Mi unidad* y pone esa ruta en `CARPETA_DRIVE`.
- **Tareas programadas:** Colab no ejecuta tareas automáticas; la extracción diaria corre en GitHub Actions (`.github/workflows/extract.yml`).